<a href="https://colab.research.google.com/github/Giocrisrai/aprendizaje-refuerzo-duoc/blob/main/notebooks/05_desafio_salmonicultura.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/></a>

# 05 · DESAFÍO: Centro de cultivo de salmón — Seno de Reloncaví
**Machine Learning — Duoc UC · Trabajo en grupo**

## Contexto
Una empresa salmonera de Puerto Montt quiere reemplazar la regla que usa hoy su jefe de centro para decidir, **cada día de una temporada de 60 días**, cómo alimentar, cuándo airear y **cuándo cosechar**. Les entrega un simulador de su centro y les pide un agente de **aprendizaje por refuerzo** que gane más dinero que la regla actual, **sin aumentar la mortalidad** y que **siga funcionando cuando las condiciones cambien**.

### Lo que ya está resuelto en las clases
El centro es nuevo. Las herramientas no. Cada sección de abajo corresponde a algo que ya corrieron:

| Sección del desafío | Dónde lo vieron | Qué tienen que reconocer |
|---|---|---|
| La grilla, el −1 por paso, el zombi | Notebook 01 | Terminar pronto es una trampa si el premio está más adelante |
| A. Explorar el entorno | Líneas base del notebook 02 | Medir una política simple antes de entrenar |
| B. Diseño del estado | Técnicas 1 y 2 del notebook 03 | La observación cruda no cabe, y le falta memoria |
| C. Por qué el ingenuo cosecha el día 0 | Técnica 3 del notebook 03, la camioneta | Q en 0, más una acción que termina el episodio |
| D. Dos algoritmos y sus curvas | Técnica 4 y las métricas del notebook 02 | Mismo presupuesto, varias semillas, IC95%, eje en **pasos** |
| E. Tabla y estrés | Técnica 6 del notebook 03 | Greedy, semillas nuevas, condiciones distintas |
| Historial del operador | Técnica 5 del notebook 03 | Puede ayudar o sesgar: ese CSV es de **otro** centro |

Si una de esas frases no se puede explicar en voz alta, vuelvan a esa clase antes de escribir código. Si hace falta practicar la cuenta antes de medir, el notebook 04 tiene seis ejercicios con celda Comprobar.

### Lo que el agente ve (observación)
`(dia, biomasa_bin, o2_sensor, tendencia_precio)` — detalles en el docstring de `salmon_env.py`.

### Lo que el agente NO ve
El oxígeno real, si hay un **bloom de algas** en curso, la biomasa exacta y el precio exacto. El sensor de oxígeno informa **con retraso** y a veces falla; la señal de precio es ruidosa. El pico de precio **cambia de temporada en temporada**.

### Acciones
`0 ayuno · 1 ración normal · 2 ración alta · 3 aireación + ración normal · 4 COSECHAR (termina la temporada)`

## Reglas
1. Cada grupo usa **su código de grupo** (lo asigna el docente). Cada código genera un centro con **parámetros distintos**: lo que funciona para otro grupo no necesariamente funciona para el suyo.
2. **Presupuesto:** cada agente se entrena con **a lo más 150.000 pasos** de interacción, y el grupo tiene **1.500.000 pasos en total** (todas las corridas, semillas y pruebas, incluidas las fallidas y la búsqueda de hiperparámetros). Alcanza para unas 10 corridas completas: **planifiquen**. Se reporta `env.pasos_usados()` y una **bitácora** con lo que gastó cada experimento. `evaluar_politica(...)` no consume presupuesto.
3. La política entregada debe ser **aprendida con RL** (tabla Q, o lo que justifiquen). Las reglas escritas a mano solo se permiten como **líneas base** para comparar.
4. Pueden usar el historial `../data/historial_operador_salmon.csv` (500 temporadas del operador). **Ojo:** viene de **otro** centro, con otras condiciones.
5. La evaluación final la hace el docente con **semillas y escenarios que ustedes no conocen** (incluye condiciones distintas a las de entrenamiento).
6. Pueden usar IA generativa como apoyo, pero **cada número, gráfico y afirmación del informe debe salir de experimentos ejecutados por ustedes en su entorno**, y lo defenderán oralmente (el docente les pedirá cambiar algo en vivo y explicar el efecto).

## Entregables
1. Carpeta `entrega_<GRUPO>/` con `politica.py` (clase `Politica` con `reset()` y `actuar(obs)`) y los archivos que cargue (por ejemplo `q.npy`). Al final de este notebook hay una función que la genera.
2. Este notebook completo con las secciones **A–F** respondidas.
3. `bitacora_<GRUPO>.csv` (se genera con `registrar(...)`).

In [ ]:
#@title ⚙️ Configuración (ejecutar primero — en Colab clona el repositorio, sin subir archivos)
import os, sys, subprocess, glob

REPO_URL = "https://github.com/Giocrisrai/aprendizaje-refuerzo-duoc"
EN_COLAB = "google.colab" in sys.modules
ARCHIVOS = ("rl_utils.py", "salmon_env.py", "grilla.py", "evolucion.py")
DATOS = (
    "../data/runs_cliffwalking.csv",
    "../data/historial_inventario.csv",
    "../data/historial_operador_salmon.csv",
)

def buscar_notebooks():
    """Carpeta que contiene los .py de la clase, o None."""
    if all(os.path.exists(nombre) for nombre in ARCHIVOS):
        return os.getcwd()
    candidatos = [
        "/content/aprendizaje-refuerzo-duoc/notebooks",
        "/content/rl_aprendizaje_refuerzo/notebooks",
    ]
    candidatos += glob.glob("/content/**/notebooks", recursive=True)
    vistos = []
    for carpeta in candidatos:
        if carpeta in vistos or not os.path.isdir(carpeta):
            continue
        vistos.append(carpeta)
        if all(os.path.exists(os.path.join(carpeta, nombre)) for nombre in ARCHIVOS):
            return carpeta
    return None

if EN_COLAB and buscar_notebooks() is None:
    subprocess.run(
        ["git", "clone", "-q", "--depth", "1", REPO_URL, "/content/aprendizaje-refuerzo-duoc"],
        check=False,
    )

carpeta = buscar_notebooks()
if carpeta is None:
    raise FileNotFoundError(
        "No encuentro " + ", ".join(ARCHIVOS) + ". "
        "En Colab esta celda clona el repositorio: ahí están los módulos y data/. "
        "En el computador, abre el notebook desde la carpeta notebooks/."
    )
os.chdir(carpeta)

if EN_COLAB:
    req = os.path.abspath(os.path.join(os.getcwd(), "..", "requirements.txt"))
    if os.path.exists(req):
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", req])
try:
    import numpy, pandas, matplotlib
except ImportError as exc:
    raise ImportError("Faltan numpy, pandas o matplotlib. En local, desde la raíz del repositorio: uv sync --group dev") from exc
faltan = [ruta for ruta in DATOS if not os.path.exists(ruta)]
if faltan:
    raise FileNotFoundError("Faltan datos del repositorio: " + ", ".join(faltan))
print("✅ Listo. Carpeta de trabajo:", os.getcwd(), "| Colab" if EN_COLAB else "| local")
print("Librerías:", "numpy", numpy.__version__, "| pandas", pandas.__version__, "| matplotlib", matplotlib.__version__)


In [ ]:
GRUPO = "G01"   # <-- CAMBIAR por el código asignado a su grupo

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import salmon_env as se
from salmon_env import SalmonFarmEnv, PoliticaAleatoria, PoliticaOperador, evaluar_politica, NOMBRES_ACCIONES

plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.3})
env = SalmonFarmEnv(GRUPO, seed=0)
print("Grupo:", env.grupo, "| presupuesto total:", se.PRESUPUESTO_PASOS, "| por corrida:", se.PASOS_POR_CORRIDA, "| usados:", env.pasos_usados())

### Bitácora de experimentos (obligatoria)
Registren **cada** experimento, también los que no funcionaron. La bitácora es parte importante de la nota: muestra cómo iteraron.

In [ ]:
BITACORA = Path(f"bitacora_{GRUPO}.csv")

def registrar(experimento, pasos_gastados, resultado, conclusion):
    fila = pd.DataFrame([dict(fecha=pd.Timestamp.now().strftime("%Y-%m-%d %H:%M"), experimento=experimento,
                              pasos_gastados=pasos_gastados, pasos_acumulados=env.pasos_usados(),
                              resultado=resultado, conclusion=conclusion)])
    fila.to_csv(BITACORA, mode="a", header=not BITACORA.exists(), index=False)
    return pd.read_csv(BITACORA).tail(3)

## Líneas base (a superar)

In [ ]:
base = {}
for nombre, pol in [("aleatoria", PoliticaAleatoria()), ("operador", PoliticaOperador())]:
    r = evaluar_politica(pol, GRUPO, n_episodios=200, seed=10_000)
    base[nombre] = r
    print(f"{nombre:10s} retorno {r['retorno_medio']:7.1f} ± {r['ic95']:5.1f} | p10 {r['p10']:7.1f} "
          f"| día cosecha {r['dia_cosecha_medio']:5.1f} | días con mortalidad {r['dias_mortalidad_medio']:.2f}")

## Punto de partida (que NO funciona)
Este es el Q-learning "de libro" que escribiría cualquiera en 5 minutos (o que les daría un chatbot). Ejecútenlo, mírenlo fallar y **diagnostiquen por qué** antes de arreglar nada: esa explicación es la sección C.

In [ ]:
def q_learning_ingenuo(pasos=20_000, seed=0, alpha=0.1, gamma=0.99):
    rng = np.random.default_rng(seed)
    e = SalmonFarmEnv(GRUPO, seed=seed)
    Q = {}                                         # tabla con la observación cruda como clave
    q = lambda s: Q.setdefault(s, np.zeros(5))
    usados, retornos = 0, []
    while usados < pasos:
        eps = max(0.05, 1 - usados / pasos)
        s, done, G = e.reset(), False, 0.0
        while not done:
            a = int(rng.integers(5)) if rng.random() < eps else int(np.argmax(q(s)))
            s2, r, done, _ = e.step(a); usados += 1; G += r
            q(s)[a] += alpha * (r + (0 if done else gamma * q(s2).max()) - q(s)[a])
            s = s2
        retornos.append(G)
    return Q, retornos

class PoliticaTabla:
    def __init__(self, Q): self.Q = Q
    def reset(self): pass
    def actuar(self, obs): return int(np.argmax(self.Q.get(obs, np.zeros(5))))

antes = env.pasos_usados()
Q_ing, ret_ing = q_learning_ingenuo()
r = evaluar_politica(PoliticaTabla(Q_ing), GRUPO, 200)
print(f"Ingenuo: retorno {r['retorno_medio']:.1f} | día de cosecha medio {r['dia_cosecha_medio']:.1f} | estados vistos {len(Q_ing)}")
registrar("Q-learning ingenuo (obs cruda, Q0=0)", env.pasos_usados() - antes, round(r["retorno_medio"], 1), "COMPLETAR: ¿por qué falla?")

---
En clase ya corrieron el ejemplo: el Q-learning ingenuo cosecha el día 0. **De aquí en adelante lo resuelven ustedes.** Este notebook no trae la solución.

### Para consultar mientras trabajan
| Si se traban en | Vuelvan a |
|---|---|
| El paso que cuesta y la acción que termina el episodio | Notebook 01, y [Q-learning con la tabla a la vista](https://www.youtube.com/watch?v=qhRNvCVVJaA) |
| Semillas, IC95% y entrenamiento distinto de evaluación | Notebook 02, y [StatQuest, 18 min](https://www.youtube.com/watch?v=Z-T0iJEXiwM) |
| SARSA y Q-learning, en español | [Lección 14](https://codificandobits.com/curso/aprendizaje-por-refuerzo-nivel-intermedio/14-control-sarsa/) y [lección 15](https://codificandobits.com/curso/aprendizaje-por-refuerzo-nivel-intermedio/15-q-learning/) |
| Monte Carlo contra TD | [David Silver, lecture 4](https://www.youtube.com/watch?v=PnHCvfgC_ZA) y [Monte Carlo en Codificando Bits](https://codificandobits.com/curso/aprendizaje-por-refuerzo-nivel-intermedio/10-control-monte-carlo/) |

## A. Exploración del entorno (descubrimiento con datos)
Respondan con **experimentos propios** (tablas o gráficos; políticas simples están permitidas aquí). Cada respuesta debe indicar cuántos pasos de presupuesto consumió.

1. ¿Alrededor de qué día conviene cosechar en **su** centro? ¿Qué tanto varía entre temporadas?
2. Cuando el oxígeno cae, ¿cuántos días suele durar el episodio? ¿Qué acción conviene durante ese período (airear, ayunar, seguir)? ¿Cuánto cuesta equivocarse?
3. ¿La ración alta conviene alguna vez? ¿Bajo qué condiciones?
4. ¿Qué tan confiables son el sensor de O₂ y la señal de precio? (Pista: compare lo que el sensor dice un día con lo que pasa con la mortalidad al día siguiente; compare la tendencia observada con el día en que el precio realmente baja.)
5. ¿El historial del operador (otro centro) se parece a su centro? Muestren evidencia (por ejemplo, frecuencia de O₂ bajo, retorno por día de cosecha).

In [ ]:
# A.0 — Para empezar: el historial del operador (otro centro)
hist = pd.read_csv("../data/historial_operador_salmon.csv")
print(hist.shape)
hist.head()

In [ ]:
# A.1 — Su código aquí

## B. Diseño del estado
- ¿Qué features usan y con qué resolución? ¿Cuántos estados resultan?
- ¿Qué **memoria** agregaron para compensar lo que la observación no muestra? Justifiquen con un experimento (mismo algoritmo y presupuesto, con y sin cada feature, ≥ 3 semillas).

## C. Diagnóstico del agente ingenuo
Expliquen **con evidencia** (no con teoría general) por qué el Q-learning ingenuo falla en su centro. Muestren al menos un gráfico o tabla que lo demuestre.

## D. Algoritmos y entrenamiento
- Comparen **al menos dos** enfoques (por ejemplo: Q-learning con buena inicialización, Monte Carlo, SARSA, pre-entrenamiento con el historial, distintas formas de explorar la acción de cosecha...) **con el mismo presupuesto**.
- ≥ 5 semillas por configuración final, curvas de aprendizaje con IC95% (en función de **pasos**, no solo de episodios).
- Indiquen cuántos pasos del presupuesto consumió cada configuración.

## E. Evaluación
- Tabla final contra aleatoria y operador: retorno medio ± IC95%, percentil 10, día medio de cosecha, días con mortalidad.
- **Pruebas de estrés propias:** diseñen al menos **2** escenarios distintos al nominal (pista: `evaluar_politica(..., params=...)` acepta parámetros modificados; vean `se.parametros_grupo(GRUPO)` **solo para construir los escenarios de estrés**) y discutan los resultados.

## F. Conclusiones y recomendación a la empresa
¿Recomendarían usar su agente en producción? ¿Con qué riesgos y qué monitorearían? (máx. 1 página)

In [ ]:
# Espacio de trabajo B–E (agreguen las celdas que necesiten)

---
## Exportar la entrega
Adapten `features` y la clase `Politica` a su diseño. **Prueben** la carpeta exportada antes de entregar con la última celda.

In [ ]:
PLANTILLA = """
import numpy as np
from pathlib import Path

_DIR = Path(__file__).parent

class Politica:
    def __init__(self):
        self.Q = np.load(_DIR / "q.npy")
        # agreguen aquí lo que necesiten cargar

    def reset(self):
        # reinicien aquí la memoria que usen (contadores, observación anterior, etc.)
        self.memoria = None

    def features(self, obs):
        # REEMPLAZAR por su diseño de estado (sección B).
        # IMPORTANTE: debe ser exactamente la misma función que usaron al entrenar,
        # incluida la memoria que actualicen aquí (y reinicien en reset()).
        return tuple(obs)

    def actuar(self, obs):
        return int(np.argmax(self.Q[self.features(obs)]))
"""

def exportar(Q, codigo_politica=PLANTILLA):
    carpeta = Path(f"entrega_{GRUPO}")
    carpeta.mkdir(exist_ok=True)
    np.save(carpeta / "q.npy", Q)
    (carpeta / "politica.py").write_text(codigo_politica.strip() + "\n", encoding="utf-8")
    return carpeta

In [ ]:
# Prueba de la entrega (descomentar cuando exporten)
# import importlib.util, sys
# carpeta = exportar(MI_Q)
# spec = importlib.util.spec_from_file_location("pol", carpeta / "politica.py"); m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m)
# print(evaluar_politica(m.Politica(), GRUPO, 200, seed=55_555))
print("Pasos usados en esta sesión:", env.pasos_usados(), "de", se.PRESUPUESTO_PASOS)

### Empaquetar la entrega
Genera `entrega_<GRUPO>.zip` con la carpeta exportada y la bitácora. **En Colab** se descarga solo (los archivos de Colab se borran al cerrar la sesión: descárguenlo siempre). Súbanlo junto con este notebook (Archivo → Descargar → .ipynb).

In [ ]:
import shutil, sys
carpeta = Path(f"entrega_{GRUPO}")
if not carpeta.exists():
    print("Primero exporten su política con exportar(...).")
else:
    if BITACORA.exists():
        shutil.copy(BITACORA, carpeta / BITACORA.name)
    import zipfile
    archivo = f"entrega_{GRUPO}.zip"
    with zipfile.ZipFile(archivo, "w", zipfile.ZIP_DEFLATED) as z:
        for f in carpeta.rglob("*"):
            if f.is_file() and "__pycache__" not in f.parts:
                z.write(f, f.relative_to("."))
    print("Creado:", archivo)
    if "google.colab" in sys.modules:
        from google.colab import files
        files.download(archivo)